# Cleaning Messy Data for Better Decisions: A Practical Pandas Case Study

Messy data is not a technical inconvenience. It is a decision risk.

A duplicate campaign row can inflate performance. A broken date can move a signal into the wrong quarter. A category typo can split one product into three fake segments. An unvalidated outlier can turn a reasonable recommendation into a bad bet.

This notebook uses a realistic business-style extract built from public Google Trends snapshots. The source signal is simple; the messiness represents common spreadsheet, CRM, and analytics-export problems that appear before data reaches a dashboard or product review.

## Kaggle publication angle

This notebook is useful because it focuses on the work analysts actually do before decisions are made:

- detect data quality risks;
- clean without hiding assumptions;
- validate the final table;
- translate cleaning choices into better business interpretation.

It is designed for analysts, students, and product teams who need a reusable pandas workflow that improves decision quality, not just code style.

## Setup

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titleweight"] = "bold"

## Load the public signal dataset

The notebook runs on Kaggle and locally. On Kaggle, attach the **Weak Signals Dataset — Google Trends France** dataset. Locally, the notebook falls back to the repository package or original snapshots.

In [ ]:
def find_dataset_file(pattern: str) -> Path:
    search_roots = [
        Path("/kaggle/input"),
        Path.cwd() / "kaggle" / "google_trends_dataset",
        Path.cwd().parent / "kaggle" / "google_trends_dataset",
        Path.cwd() / "data" / "snapshots",
        Path.cwd().parent / "data" / "snapshots",
    ]
    for root in search_roots:
        if root.exists():
            matches = sorted(root.rglob(pattern))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Could not find {pattern}.")

source_path = find_dataset_file("iot_FR_today_5-y_chatgpt_iphone_meteo.csv")
source = pd.read_csv(source_path)
source.head()

## Create a realistic messy business extract

The public source is clean. Real business extracts often are not. To make the cleaning workflow concrete, we convert the time-series snapshot into a long-format reporting extract and introduce common export issues transparently:

- duplicate rows;
- inconsistent product/category labels;
- mixed date formats;
- numeric values stored as text;
- missing values;
- one impossible outlier.

This keeps the signal grounded in real public data while making the cleaning case study realistic.

In [ ]:
def build_messy_business_extract(clean_source: pd.DataFrame) -> pd.DataFrame:
    raw = clean_source.copy()
    raw["date"] = pd.to_datetime(raw["date"])
    recent = raw.tail(36).copy()

    long = recent.melt(id_vars="date", var_name="product_signal", value_name="interest_score")
    long["market"] = "FR"
    long["source"] = "Google Trends"
    long["report_owner"] = np.where(long["product_signal"].eq("chatgpt"), "AI product", "Growth")

    messy = long.copy().reset_index(drop=True)

    # Spreadsheet-style category inconsistencies.
    messy.loc[messy.index % 17 == 0, "product_signal"] = messy.loc[messy.index % 17 == 0, "product_signal"].str.upper()
    messy.loc[messy.index % 19 == 0, "product_signal"] = messy.loc[messy.index % 19 == 0, "product_signal"] + " "
    messy.loc[messy.index % 23 == 0, "market"] = "France"
    messy.loc[messy.index % 29 == 0, "market"] = "fr"

    # Mixed date formats from manual exports.
    date_values = messy["date"].copy()
    messy["date"] = date_values.dt.strftime("%Y-%m-%d")
    messy.loc[messy.index % 13 == 0, "date"] = date_values.loc[messy.index % 13 == 0].dt.strftime("%d/%m/%Y")
    messy.loc[messy.index % 31 == 0, "date"] = date_values.loc[messy.index % 31 == 0].dt.strftime("%b %d, %Y")

    # Numeric values stored as text, including blanks and one impossible value.
    messy["interest_score"] = messy["interest_score"].astype(str)
    messy.loc[messy.index % 11 == 0, "interest_score"] = messy.loc[messy.index % 11 == 0, "interest_score"] + "%"
    messy.loc[messy.index % 37 == 0, "interest_score"] = "n/a"
    messy.loc[7, "interest_score"] = "999"

    # Duplicate export rows.
    messy = pd.concat([messy, messy.iloc[[4, 12, 20]]], ignore_index=True)
    return messy.sample(frac=1, random_state=42).reset_index(drop=True)

raw_extract = build_messy_business_extract(source)
raw_extract.head(10)

In [ ]:
raw_extract.shape

## First inspection: where decisions can go wrong

Before cleaning, inspect the table as a decision artifact. The question is not only “what is dirty?” but “what business conclusion could this distort?”

In [ ]:
profile = pd.DataFrame({
    "dtype": raw_extract.dtypes.astype(str),
    "missing": raw_extract.isna().sum(),
    "unique_values": raw_extract.nunique(dropna=False),
})
profile

In [ ]:
raw_extract.duplicated().sum()

## Missing values

Missing values are not automatically errors. They become risks when the missingness changes rankings, averages, or trend interpretation.

In [ ]:
missing_tokens = ["", "na", "n/a", "none", "null", "missing"]
missing_like = raw_extract.apply(lambda col: col.astype(str).str.strip().str.lower().isin(missing_tokens).sum())
missing_like.to_frame("missing_like_tokens")

## Duplicate handling

Duplicate rows can double-count a signal. For a business review, that can exaggerate a trend or make one segment look more important than it is.

In [ ]:
duplicate_rows = raw_extract[raw_extract.duplicated(keep=False)].sort_values(list(raw_extract.columns))
duplicate_rows

In [ ]:
deduped = raw_extract.drop_duplicates().copy()
print("Rows before:", len(raw_extract))
print("Rows after duplicate removal:", len(deduped))

## Inconsistent categories

Category cleanup should be explicit and auditable. Silent replacements are dangerous when categories carry business meaning.

In [ ]:
category_snapshot = {
    "product_signal": sorted(deduped["product_signal"].astype(str).unique()),
    "market": sorted(deduped["market"].astype(str).unique()),
}
category_snapshot

In [ ]:
category_map = {
    "chatgpt": "chatgpt",
    "iphone": "iphone",
    "meteo": "meteo",
}
market_map = {
    "fr": "FR",
    "france": "FR",
}

clean = deduped.copy()
clean["product_signal"] = clean["product_signal"].astype(str).str.strip().str.lower().map(category_map)
clean["market"] = clean["market"].astype(str).str.strip().str.lower().map(market_map)

clean[["product_signal", "market"]].drop_duplicates().sort_values(["product_signal", "market"])

## Date parsing issues

Mixed date formats are common in exports. We parse deliberately and check for failures instead of assuming the conversion worked.

In [ ]:
def parse_mixed_date(value):
    text = str(value).strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%b %d, %Y"):
        parsed = pd.to_datetime(text, format=fmt, errors="coerce")
        if pd.notna(parsed):
            return parsed
    return pd.NaT

clean["date"] = clean["date"].apply(parse_mixed_date)
clean["date_parse_failed"] = clean["date"].isna()
clean["date_parse_failed"].sum()

## Numeric cleaning

Scores arrive as strings, percentages, blanks, or impossible values. The cleaning rule is simple: extract the number, validate the expected range, and keep a flag for suspicious rows.

In [ ]:
def parse_score(value):
    text = str(value).strip().lower()
    if text in {"", "na", "n/a", "none", "null"}:
        return np.nan
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(match.group(0)) if match else np.nan

clean["interest_score_raw"] = clean["interest_score"]
clean["interest_score"] = clean["interest_score"].apply(parse_score)
clean[["interest_score_raw", "interest_score"]].head(12)

## Outlier detection

For Google Trends, valid values should be between 0 and 100. A score of 999 is not an exceptional market event; it is a data quality defect.

In [ ]:
clean["score_out_of_range"] = clean["interest_score"].lt(0) | clean["interest_score"].gt(100)
clean.loc[clean["score_out_of_range"], ["date", "product_signal", "market", "interest_score_raw", "interest_score"]]

In [ ]:
clean.loc[clean["score_out_of_range"], "interest_score"] = np.nan

# For this reporting extract, impute missing scores by signal median.
signal_median = clean.groupby("product_signal")["interest_score"].transform("median")
clean["interest_score"] = clean["interest_score"].fillna(signal_median)
clean["interest_score"] = clean["interest_score"].round(1)
clean.head()

## Validation checks

A clean dataset is not “whatever pandas accepts.” It must satisfy business rules that protect downstream interpretation.

In [ ]:
required_columns = ["date", "product_signal", "market", "source", "report_owner", "interest_score"]
final = clean[required_columns].copy()
final = final.sort_values(["date", "product_signal"]).reset_index(drop=True)

checks = {
    "no_missing_required_values": bool(final[required_columns].notna().all().all()),
    "unique_signal_date_market": bool(not final.duplicated(["date", "product_signal", "market"]).any()),
    "scores_between_0_and_100": bool(final["interest_score"].between(0, 100).all()),
    "known_signals_only": bool(set(final["product_signal"]).issubset({"chatgpt", "iphone", "meteo"})),
    "known_market_only": bool(set(final["market"]) == {"FR"}),
}
checks

In [ ]:
assert all(checks.values()), checks
final.head(10)

## Final clean dataset

The final table is intentionally narrow and decision-ready: one row per date, signal, and market.

In [ ]:
final_summary = final.groupby("product_signal").agg(
    rows=("interest_score", "size"),
    first_date=("date", "min"),
    last_date=("date", "max"),
    average_score=("interest_score", "mean"),
    median_score=("interest_score", "median"),
    max_score=("interest_score", "max"),
).round(2)
final_summary

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=final, x="date", y="interest_score", hue="product_signal", marker="o", ax=ax)
ax.set_title("Cleaned business signal extract")
ax.set_xlabel("")
ax.set_ylabel("Google Trends index")
ax.legend(title="Signal")
plt.show()

## Business interpretation

Cleaning changes the quality of the conversation:

- duplicates no longer inflate repeated rows;
- category variants no longer split the same signal;
- broken dates no longer move observations into the wrong period;
- impossible scores no longer dominate the chart;
- validation checks make the dataset safe enough for a decision review.

The cleaned dataset supports a sharper question: which signal deserves attention now, and what decision should follow?

In [ ]:
latest = final.sort_values("date").groupby("product_signal").tail(1).set_index("product_signal")["interest_score"]
median = final.groupby("product_signal")["interest_score"].median()
decision_table = pd.DataFrame({
    "latest_score": latest,
    "median_score": median,
})
decision_table["latest_vs_median"] = decision_table["latest_score"] - decision_table["median_score"]
decision_table.sort_values("latest_vs_median", ascending=False).round(2)

## Decision framing

A practical decision rule:

- If a signal is above its recent median, investigate whether the shift is meaningful.
- If a signal is stable and seasonal, plan operations around it rather than treating it as news.
- If a signal is volatile, avoid acting on one point; require context or a second source.

For product teams, the recommendation is not “trust the chart.” The recommendation is “clean the evidence before the meeting, then use the chart to ask better questions.”

## Limitations

This notebook uses a controlled messy extract built from public Google Trends snapshots. That makes the workflow reproducible, but real organizations may face additional issues: joins across systems, currency formats, user-level privacy constraints, schema drift, and conflicting business definitions.

Google Trends itself also remains a relative attention index, not a measure of sales, adoption, or causality.

## Conclusion

Good cleaning is not cosmetic. It protects decisions.

This case study shows a reusable workflow:

1. inspect quality risks;
2. remove duplicates;
3. standardize categories;
4. parse dates deliberately;
5. clean numeric fields;
6. detect impossible values;
7. validate the final table;
8. translate the clean dataset into a decision frame.

That is the difference between a chart that looks convincing and evidence that is ready for a business decision.